# 05 ML GNN Embeddings

This notebook uses `MLTrainAndStore` to train regressors with only the GraphSAGE embedding features.

In [1]:
from pathlib import Path
import sys
import os

sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
import xgboost as xgb
from xgboost import XGBRegressor

from src.models.ml_train_and_store import (
    ModelTrainer,
    load_gnn_dataset,
    make_pipeline,
)

pd.set_option("display.max_columns", 200)
PROJECT_ROOT = Path().resolve().parents[1]

In [2]:
print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis


## Load Dataset

In [3]:
df, feature_cols = load_gnn_dataset(PROJECT_ROOT, filename="graphsage_srisk_dataset.parquet")
print(df.shape, len(feature_cols))

df_1, feature_cols_1 = load_gnn_dataset(PROJECT_ROOT, target_col="log_systemic_risk_label", filename="node2vec_srisk_dataset.parquet")
print(df_1.shape, len(feature_cols_1))

(145536, 69) 64
(145536, 69) 64


In [4]:
trainer = ModelTrainer(
    df=df,
    feature_cols=feature_cols,
    target_col="log_systemic_risk_label",
)

trainer.train_df.shape, trainer.val_df.shape, trainer.test_df.shape

((109152, 69), (18192, 69), (13644, 69))

In [5]:
trainer_1 = ModelTrainer(
    df=df_1,
    feature_cols=feature_cols_1,
    target_col="log_systemic_risk_label",
)

trainer_1.train_df.shape, trainer_1.val_df.shape, trainer_1.test_df.shape

((109152, 69), (18192, 69), (13644, 69))

## Define Models

In [6]:
candidate_models = {
    "linear_regression": make_pipeline(LinearRegression(), scale_features=True),
}

list(candidate_models)

['linear_regression']

## Train And Store

In [7]:
trainer.train_all(candidate_models)

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,linear_regression,0.040886,0.057922,0.060012,0.152222,0.23399,0.218862,0.257071,0.018872,-0.1852


In [8]:
trainer_1.train_all(candidate_models)

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,linear_regression,0.046205,0.056368,0.0517,0.163722,0.245072,0.196954,0.140576,-0.076262,0.0402


In [9]:
trainer.leaderboard()

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,linear_regression,0.040886,0.057922,0.060012,0.152222,0.23399,0.218862,0.257071,0.018872,-0.1852


In [10]:
trainer_1.leaderboard()

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,linear_regression,0.046205,0.056368,0.0517,0.163722,0.245072,0.196954,0.140576,-0.076262,0.0402


## Single-Model Pattern

In [11]:
amodel = make_pipeline(
    XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42),
    scale_features=True,
)

trainer.train(model=amodel, name="XGBRegressor_search_1")
trainer.leaderboard()

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,XGBRegressor_search_1,0.010979,0.035095,0.028452,0.063875,0.227482,0.194681,0.869185,0.072692,0.062222
1,linear_regression,0.040886,0.057922,0.060012,0.152222,0.23399,0.218862,0.257071,0.018872,-0.1852


In [12]:
trainer_1.train(model=amodel, name="XGBRegressor_search_1")
trainer_1.leaderboard()

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,XGBRegressor_search_1,0.013023,0.036682,0.028801,0.062261,0.205007,0.168002,0.875712,0.246877,0.30164
1,linear_regression,0.046205,0.056368,0.0517,0.163722,0.245072,0.196954,0.140576,-0.076262,0.0402


## Best Model

In [13]:
trainer.best_name()

'XGBRegressor_search_1'

In [14]:
trainer_1.best_name()

'XGBRegressor_search_1'

In [15]:
trainer.test_predictions().head(20)

,bank_id,year,quarter,period,log_systemic_risk_label,prediction,abs_error
0,8,2023,1,2023Q1,3.988984,1.040853,2.948131
1,17,2023,1,2023Q1,3.806662,1.194091,2.612571
2,5,2023,1,2023Q1,4.290459,1.708197,2.582262
3,7,2023,1,2023Q1,3.610918,1.215647,2.395270
4,6,2023,2,2023Q2,3.610918,1.265854,2.345064
5,0,2023,1,2023Q1,4.043051,1.742119,2.300932
6,5,2023,2,2023Q2,4.110874,1.815828,2.295046
7,4228,2023,2,2023Q2,0.693147,2.964937,2.271790
8,28,2023,2,2023Q2,3.091042,0.825819,2.265224
9,5,2023,3,2023Q3,4.189655,1.981370,2.208285


In [16]:
trainer_1.test_predictions().head(20)

,bank_id,year,quarter,period,log_systemic_risk_label,prediction,abs_error
0,1,2023,1,2023Q1,3.737670,0.929773,2.807896
1,8,2023,1,2023Q1,3.988984,1.193291,2.795693
2,17,2023,1,2023Q1,3.806662,1.021158,2.785504
3,0,2023,1,2023Q1,4.043051,1.325601,2.717450
4,6,2023,1,2023Q1,3.761200,1.131382,2.629818
5,2,2023,1,2023Q1,3.713572,1.101144,2.612428
6,6,2023,2,2023Q2,3.610918,1.021060,2.589858
7,5,2023,3,2023Q3,4.189655,1.665936,2.523719
8,17,2023,2,2023Q2,3.637586,1.187078,2.450508
9,2,2023,2,2023Q2,3.465736,1.033189,2.432547
